<a href="https://colab.research.google.com/github/edwardsnj/glygen-colab-notebooks/blob/main/variants/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# REPO_BASE = "https://raw.githubusercontent.com/edwardsnj/glygen-colab-notebooks/refs/heads/main"
# NOTEBOOK = "variants"

# import httpimport
# with httpimport.remote_repo(REPO_BASE + "/" + NOTEBOOK):
#   import make_plotdata, map_datasets, run_binomial_test

# import necessary modules
import pandas as pd
import urllib, os.path

# create directories if necessary
for dirname in ("data","reviewed"):
  if not os.path.exists(dirname):
    os.mkdir(dirname)

# GlyGen review datafiles
GLYGEN_DATA_REVIEWED = "https://data.glygen.org/ln2data/releases/data/current/reviewed"

human_protein_mutation_files = """
  human_protein_mutation_cancer_all.csv
  human_protein_mutation_germline_all.csv
"""

human_proteoform_glycosylation_site_files = """
  human_proteoform_glycosylation_sites_carbbank.csv
  human_proteoform_glycosylation_sites_c_man.csv
  human_proteoform_glycosylation_sites_diabetes_glycomic.csv
  human_proteoform_glycosylation_sites_embl.csv
  human_proteoform_glycosylation_sites_glycomeatlas.csv
  human_proteoform_glycosylation_sites_glyconnect.csv
  human_proteoform_glycosylation_sites_gptwiki.csv
  human_proteoform_glycosylation_sites_harvard.csv
  human_proteoform_glycosylation_sites_literature.csv
  human_proteoform_glycosylation_sites_literature_mining.csv
  human_proteoform_glycosylation_sites_literature_mining_manually_verified.csv
  human_proteoform_glycosylation_sites_oglcnac_atlas.csv
  human_proteoform_glycosylation_sites_oglcnac_mcw.csv
  human_proteoform_glycosylation_sites_o_gluc.csv
  human_proteoform_glycosylation_sites_o_gluc_predicted.csv
  human_proteoform_glycosylation_sites_pdb.csv
  human_proteoform_glycosylation_sites_pdc_ccrcc.csv
  human_proteoform_glycosylation_sites_platelet.csv
  human_proteoform_glycosylation_sites_predicted_isoglyp.csv
  human_proteoform_glycosylation_sites_rcsb_pdb.csv
  human_proteoform_glycosylation_sites_tablemaker.csv
  human_proteoform_glycosylation_sites_twinsuk.csv
  human_proteoform_glycosylation_sites_tyr_o_linked.csv
  human_proteoform_glycosylation_sites_unicarbkb.csv
  human_proteoform_glycosylation_sites_unicarbkb_glycomics_study.csv
  human_proteoform_glycosylation_sites_uniprotkb.csv
"""

In [4]:
# Create data frames for experimental and predicted glycosites

if not os.path.exists("data/glyco_sites_exp.csv") or \
   not os.path.exists("data/glyco_sites_pred.csv"):

  dfs = []
  for fn in filter(lambda fn: "_uniprotkb" not in fn,human_proteoform_glycosylation_site_files.split()):
    print(f"Download {fn}...", end="")
    df = pd.read_csv(GLYGEN_DATA_REVIEWED + "/" + fn,
                    usecols=["uniprotkb_canonical_ac",
                              "start_pos","start_aa",
                              "glycosylation_type"])
    print(f" done ({df.shape[0]} rows).")
    df = df[~df["uniprotkb_canonical_ac"].isna() & ~df["start_pos"].isna()]
    df['start_pos'] = df['start_pos'].astype(int)
    df["predicted"] = ("_predicted_" in fn)
    dfs.append(df)

  exp_xref_keys = set(["protein_xref_pubmed","protein_xref_doi"])
  for fn in filter(lambda fn: "_uniprotkb" in fn,human_proteoform_glycosylation_site_files.split()):
    print(f"Download {fn}...", end="")
    df = pd.read_csv(GLYGEN_DATA_REVIEWED + "/" + fn,
                    usecols=["uniprotkb_canonical_ac",
                              "start_pos","start_aa",
                              "glycosylation_type",
                              "xref_key"])
    print(f" done ({df.shape[0]} rows).")
    df = df[~df["uniprotkb_canonical_ac"].isna() & ~df["start_pos"].isna()]
    df['start_pos'] = df['start_pos'].astype(int)
    df["predicted"] = df['xref_key'].isin(exp_xref_keys)
    df = df.drop(columns=['xref_key'])
    dfs.append(df)

  glyco_sites = pd.concat(dfs,ignore_index=True)

  glyco_sites_exp = glyco_sites[~glyco_sites['predicted']]
  glyco_sites_exp = glyco_sites_exp.drop_duplicates()
  glyco_sites_exp = glyco_sites.drop(columns=['predicted'])

  glyco_sites_pred = glyco_sites[glyco_sites['predicted']]
  glyco_sites_pred = glyco_sites_pred.drop_duplicates()
  glyco_sites_pred = glyco_sites_pred.drop(columns=['predicted'])

  glyco_sites_exp.to_csv("data/glyco_sites_exp.csv",index=False)
  print(f"Wrote data/glyco_sites_exp.csv: {glyco_sites_exp.shape[0]} rows.")

  glyco_sites_pred.to_csv("data/glyco_sites_pred.csv",index=False)
  print(f"Wrote data/glyco_sites_pred.csv: {glyco_sites_pred.shape[0]} rows.")

  del dfs, df, fn, exp_xref_keys, glyco_sites

else:

  glyco_sites_pred = pd.read_csv("data/glyco_sites_pred.csv")
  print(f"Read data/glyco_sites_pred.csv: {glyco_sites_pred.shape[0]} rows.")
  glyco_sites_exp = pd.read_csv("data/glyco_sites_exp.csv")
  print(f"Read data/glyco_sites_exp.csv: {glyco_sites_exp.shape[0]} rows.")


Read data/glyco_sites_pred.csv: 11048 rows.
Read data/glyco_sites_exp.csv: 9019512 rows.


In [5]:
# Create data-frames for missense variants
if not os.path.exists("data/missense_variants.csv"):

  dfs = []
  for fn in filter(lambda fn: "_cancer" in fn,human_protein_mutation_files.split()):
    print(f"Download {fn}...", end="")
    df = pd.read_csv(GLYGEN_DATA_REVIEWED + "/" + fn,
                    usecols=["uniprotkb_canonical_ac",
                              "aa_pos","ref_aa","alt_aa",
                              "do_name"])
    print(f" done ({df.shape[0]} rows).")
    df['variant_type'] = 'somatic_cancer'
    df['aa_pos'] = df['aa_pos'].astype(int)
    df = df[df["aa_pos"]>1]
    df = df[df["ref_aa"] != df["alt_aa"]]
    df['dstatus'] = (~df["do_name"].isna())
    df.drop(columns=['do_name'],inplace=True)
    dfs.append(df)

  for fn in filter(lambda fn: "_cancer" not in fn,human_protein_mutation_files.split()):
    print(f"Download {fn}...",end="")
    nrows = 0
    if not os.path.exists("reviewed/"+fn):
      urllib.request.urlretrieve(GLYGEN_DATA_REVIEWED + "/" + fn,"reviewed/"+fn)
    for df in pd.read_csv("reviewed/" + fn,
                          usecols=["uniprotkb_canonical_ac",
                              "begin_aa_pos","end_aa_pos",
                              "ref_aa","alt_aa",
                              "do_id","mim_id"],
                          chunksize=1000000):
      df['variant_type'] = 'germline'
      df['begin_aa_pos'] = df['begin_aa_pos'].astype(int)
      df['end_aa_pos'] = df['end_aa_pos'].astype(int)
      df = df[df["begin_aa_pos"]>1]
      df = df[df['begin_aa_pos'] == df['end_aa_pos']]
      df = df[df["ref_aa"] != df["alt_aa"]]
      df['aa_pos'] = df['begin_aa_pos']
      df['dstatus'] = (~df["do_id"].isna() & ~df["mim_id"].isna())
      df.drop(columns=['begin_aa_pos','end_aa_pos','do_id','mim_id'],inplace=True)
      df = df.drop_duplicates()
      nrows += df.shape[0]
      dfs.append(df)
    print(f" done ({nrows} rows).")

  missense_variants = pd.concat(dfs,ignore_index=True)
  missense_variants = missense_variants.drop_duplicates()

  missense_variants.to_csv("data/missense_variants.csv",index=False)
  print(f"Wrote data/missense_variants.csv: {missense_variants.shape[0]} rows.")

  del dfs, df, fn

else:

  missense_variants = pd.read_csv("data/missense_variants.csv")
  print(f"Read data/missense_variants.csv: {missense_variants.shape[0]} rows.")

Read data/missense_variants.csv: 14594423 rows.


In [ ]:
glyco_type = 'N-linked'
var_type = 'somatic_cancer'
site_type = 'experimental'

if site_type == 'experimental':
  glyco_sites = glyco_sites_exp[glyco_sites_exp['glycosylation_type']==glyco_type]
else:
  glyco_sites = glyco_sites_pred[glyco_sites_pred['glycosylation_type']==glyco_type]
variants = missense_variants[missense_variants['variant_type']==var_type]
print(f"Glycosites: {glyco_sites.shape[0]}")
print(f"Variants: {variants.shape[0]}")

alluniprotkb = sorted(set(glyco_sites['uniprotkb_canonical_ac']) | set(variants['uniprotkb_canonical_ac']))
print(f"UniProtKB: {len(alluniprotkb)}")

chunksize = 1000
for chunkstart in range(0,len(alluniprotkb),chunksize):
  merged = pd.merge(glyco_sites[glyco_sites['uniprotkb_canonical_ac']].isin(alluniprotkb[chunkstart:(chunkstart+chunksize)]),
                    variants[variants['uniprotkb_canonical_ac']].isin(alluniprotkb[chunkstart:(chunkstart+chunksize)]),
                    on='uniprotkb_canonical_ac',suffixes=('_glyco','_var'))
  print(merged)
  break



Glycosites: 8716241
Variants: 3647883
